# Silver Layer ETL Pipeline

This notebook demonstrates the Silver-layer ETL process for the insurance fraud analytics project.

## Objectives
- Load raw IEEE-CIS and Porto Seguro tables from Databricks
- Apply cleaning logic based on Day 2 EDA
- Drop high-null IEEE-CIS columns
- Impute remaining missing values
- Handle Porto Seguro `-1` encoded missing values
- Validate cleaned outputs

In [0]:
import pandas as pd
import numpy as np

In [0]:
txn = spark.table("default.train_transaction")
idn = spark.table("default.train_identity")
porto = spark.table("default.train")

In [0]:
print("Raw table shapes")
print("Transactions:", txn.count(), "rows,", len(txn.columns), "cols")
print("Identity    :", idn.count(), "rows,", len(idn.columns), "cols")
print("Porto       :", porto.count(), "rows,", len(porto.columns), "cols")

Raw table shapes
Transactions: 590540 rows, 394 cols
Identity    : 144233 rows, 41 cols
Porto       : 595212 rows, 59 cols


## Sampling for notebook demonstration

The production cleaning logic was implemented in `src/etl/clean.py` and tested locally on full datasets.

For this Databricks notebook, sampled data is used for demonstration to keep the notebook lightweight and easier to run.

In [0]:
#Convert samples to pandas
df_txn = txn.sample(fraction=0.1, seed=42).toPandas()
df_idn = idn.sample(fraction=0.1, seed=42).toPandas()
df_porto = porto.sample(fraction=0.1, seed=42).toPandas()

In [0]:
print("Sampled pandas shapes")
print("df_txn   :", df_txn.shape)
print("df_idn   :", df_idn.shape)
print("df_porto :", df_porto.shape)

Sampled pandas shapes
df_txn   : (58905, 394)
df_idn   : (14480, 41)
df_porto : (59660, 59)


In [0]:
NULL_DROP_THRESHOLD = 70.0

def build_null_strategy(df: pd.DataFrame, threshold: float = NULL_DROP_THRESHOLD):
    null_pct = (df.isnull().mean() * 100).round(2)

    null_df = (
        null_pct[null_pct > 0]
        .reset_index()
        .rename(columns={"index": "column", 0: "null_pct"})
    )

    null_df["dtype"] = null_df["column"].map(df.dtypes.astype(str).to_dict())
    null_df["null_count"] = null_df["column"].map(df.isnull().sum().to_dict())
    null_df["strategy"] = np.where(null_df["null_pct"] > threshold, "Drop", "Impute")
    null_df = null_df.sort_values("null_pct", ascending=False).reset_index(drop=True)

    drop_cols = null_df.loc[null_df["strategy"] == "Drop", "column"].tolist()
    impute_cols = null_df.loc[null_df["strategy"] == "Impute", "column"].tolist()

    return drop_cols, impute_cols, null_df


def impute_dataframe(df: pd.DataFrame, exclude_cols=None):
    df = df.copy()
    exclude_cols = exclude_cols or []

    for col in df.columns:
        if col in exclude_cols:
            continue

        if df[col].isnull().sum() == 0:
            continue

        if pd.api.types.is_numeric_dtype(df[col]):
            df[col] = df[col].fillna(df[col].median())
        else:
            mode_series = df[col].mode(dropna=True)
            fill_val = mode_series.iloc[0] if not mode_series.empty else "Unknown"
            df[col] = df[col].fillna(fill_val)

    return df


def clean_ieee_transaction(df_txn: pd.DataFrame):
    df_txn = df_txn.copy()

    drop_cols, impute_cols, null_strategy_df = build_null_strategy(df_txn, NULL_DROP_THRESHOLD)

    if "isFraud" in drop_cols:
        drop_cols.remove("isFraud")

    df_txn = df_txn.drop(columns=drop_cols, errors="ignore")
    remaining_impute_cols = [c for c in impute_cols if c in df_txn.columns and c != "isFraud"]
    df_txn = impute_dataframe(df_txn, exclude_cols=["isFraud"])

    if "TransactionAmt" in df_txn.columns:
        df_txn["TransactionAmt_log1p"] = np.log1p(df_txn["TransactionAmt"])

    return df_txn, drop_cols, remaining_impute_cols, null_strategy_df


def clean_ieee_identity(df_idn: pd.DataFrame):
    df_idn = df_idn.copy()
    df_idn = impute_dataframe(df_idn, exclude_cols=["TransactionID"])
    return df_idn


def clean_porto(df_porto: pd.DataFrame):
    df_porto = df_porto.copy()

    minus1_cols = []
    for col in df_porto.columns:
        if col in ["id", "target"]:
            continue

        try:
            minus1_count = (df_porto[col] == -1).sum()
            if minus1_count > 0:
                minus1_cols.append(col)
                df_porto[col] = df_porto[col].replace(-1, np.nan)
        except Exception:
            pass

    df_porto = impute_dataframe(df_porto, exclude_cols=["id", "target"])

    return df_porto, minus1_cols

## Apply cleaning logic

The IEEE-CIS transaction sample uses Day 2 EDA rules:
- drop columns with more than 70% null values
- impute the remaining columns

The Porto Seguro sample replaces `-1` encoded missing values with nulls first, then imputes them.

In [0]:
## Run cleaning logic

df_txn_clean, drop_cols, impute_cols, null_strategy_df = clean_ieee_transaction(df_txn)
df_idn_clean = clean_ieee_identity(df_idn)
df_porto_clean, minus1_cols = clean_porto(df_porto)

/home/spark-1961feeb-8523-4450-85e5-4d/.ipykernel/2024/command-6170368145933327-1912178482:57: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_txn["TransactionAmt_log1p"] = np.log1p(df_txn["TransactionAmt"])


In [0]:
#Compare raw vs cleaned shapes
print("Before vs After cleaning")
print("IEEE txn raw   :", df_txn.shape)
print("IEEE txn clean :", df_txn_clean.shape)

print("IEEE id raw    :", df_idn.shape)
print("IEEE id clean  :", df_idn_clean.shape)

print("Porto raw      :", df_porto.shape)
print("Porto clean    :", df_porto_clean.shape)

Before vs After cleaning
IEEE txn raw   : (58905, 394)
IEEE txn clean : (58905, 227)
IEEE id raw    : (14480, 41)
IEEE id clean  : (14480, 41)
Porto raw      : (59660, 59)
Porto clean    : (59660, 59)


In [0]:
print("Dropped IEEE columns:", len(drop_cols))
print("Imputed IEEE columns:", len(impute_cols))
print("Porto -1 encoded columns handled:", len(minus1_cols))

Dropped IEEE columns: 168
Imputed IEEE columns: 174
Porto -1 encoded columns handled: 10


In [0]:
display(null_strategy_df.head(30))

column,null_pct,dtype,null_count,strategy
D7,93.68,float64,55185,Drop
dist2,93.64,float64,55158,Drop
D13,89.67,float64,52822,Drop
D14,89.58,float64,52767,Drop
D12,89.24,float64,52567,Drop
D6,87.82,float64,51733,Drop
D9,87.4,float64,51481,Drop
D8,87.4,float64,51481,Drop
V147,85.96,float64,50634,Drop
V150,85.96,float64,50634,Drop


## Validation checks

The cleaned samples are validated to confirm:
- required columns are still present
- target columns have no nulls
- duplicates are not introduced
- no unexpected null values remain

In [0]:
def validate_required_columns(df, required_cols, table_name):
    missing = [col for col in required_cols if col not in df.columns]
    if missing:
        print(f"{table_name}: missing required columns -> {missing}")
    else:
        print(f"{table_name}: required columns present.")

def validate_no_nulls_in_target(df, target_col, table_name):
    if df[target_col].isnull().sum() > 0:
        print(f"{table_name}: target column contains nulls.")
    else:
        print(f"{table_name}: target column has no nulls.")

def validate_duplicates(df, table_name):
    print(f"{table_name}: duplicate rows = {int(df.duplicated().sum())}")

def validate_remaining_nulls(df, table_name):
    print(f"{table_name}: columns with remaining nulls = {int((df.isnull().sum() > 0).sum())}")

In [0]:
print("[1] Validate IEEE transactions")
validate_required_columns(df_txn_clean, ["TransactionID", "isFraud"], "IEEE transactions")
validate_no_nulls_in_target(df_txn_clean, "isFraud", "IEEE transactions")
validate_duplicates(df_txn_clean, "IEEE transactions")
validate_remaining_nulls(df_txn_clean, "IEEE transactions")

print("\n[2] Validate IEEE identity")
validate_required_columns(df_idn_clean, ["TransactionID"], "IEEE identity")
validate_duplicates(df_idn_clean, "IEEE identity")
validate_remaining_nulls(df_idn_clean, "IEEE identity")

print("\n[3] Validate Porto Seguro")
validate_required_columns(df_porto_clean, ["id", "target"], "Porto Seguro")
validate_no_nulls_in_target(df_porto_clean, "target", "Porto Seguro")
validate_duplicates(df_porto_clean, "Porto Seguro")
validate_remaining_nulls(df_porto_clean, "Porto Seguro")

[1] Validate IEEE transactions
IEEE transactions: required columns present.
IEEE transactions: target column has no nulls.
IEEE transactions: duplicate rows = 0
IEEE transactions: columns with remaining nulls = 0

[2] Validate IEEE identity
IEEE identity: required columns present.
IEEE identity: duplicate rows = 0
IEEE identity: columns with remaining nulls = 0

[3] Validate Porto Seguro
Porto Seguro: required columns present.
Porto Seguro: target column has no nulls.
Porto Seguro: duplicate rows = 0
Porto Seguro: columns with remaining nulls = 0


## Key Silver-layer findings

- IEEE-CIS transaction columns with very high null percentages were dropped
- Remaining IEEE-CIS missing values were imputed
- Porto Seguro had no true nulls in EDA, but `-1` encoded missing values were handled
- Validation showed that required columns remained intact and no nulls remained in cleaned samples

In [0]:
display(df_txn_clean.head())
display(df_porto_clean.head())

TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,card6,addr1,addr2,dist1,P_emaildomain,C1,C2,C3,C4,C5,C6,C7,C8,C9,C10,C11,C12,C13,C14,D1,D2,D3,D4,D5,D10,D11,D15,M1,M2,M3,M4,M5,M6,M7,M8,M9,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,V29,V30,V31,V32,V33,V34,V35,V36,V37,V38,V39,V40,V41,V42,V43,V44,V45,V46,V47,V48,V49,V50,V51,V52,V53,V54,V55,V56,V57,V58,V59,V60,V61,V62,V63,V64,V65,V66,V67,V68,V69,V70,V71,V72,V73,V74,V75,V76,V77,V78,V79,V80,V81,V82,V83,V84,V85,V86,V87,V88,V89,V90,V91,V92,V93,V94,V95,V96,V97,V98,V99,V100,V101,V102,V103,V104,V105,V106,V107,V108,V109,V110,V111,V112,V113,V114,V115,V116,V117,V118,V119,V120,V121,V122,V123,V124,V125,V126,V127,V128,V129,V130,V131,V132,V133,V134,V135,V136,V137,V279,V280,V281,V282,V283,V284,V285,V286,V287,V288,V289,V290,V291,V292,V293,V294,V295,V296,V297,V298,V299,V300,V301,V302,V303,V304,V305,V306,V307,V308,V309,V310,V311,V312,V313,V314,V315,V316,V317,V318,V319,V320,V321,TransactionAmt_log1p
3504660,0,13559323,969.95,W,11942,570.0,150.0,visa,226.0,debit,310.0,87.0,15.0,gmail.com,3.0,2.0,0.0,0.0,2.0,3.0,0.0,0.0,1.0,0.0,1.0,0.0,10.0,3.0,0.0,0.0,0.0,422.0,0.0,422.0,422.0,422.0,T,T,T,M0,F,T,F,F,T,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,994.9500122070312,994.9500122070312,994.9500122070312,994.9500122070312,994.9500122070312,994.9500122070312,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,2.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,994.9500122070312,994.9500122070312,994.9500122070312,994.9500122070312,994.9500122070312,994.9500122070312,994.9500122070312,994.9500122070312,994.9500122070312,994.9500122070312,0.0,0.0,0.0,0.0,0.0,0.0,6.878274973659629
3504662,0,13559410,92.0,W,11151,307.0,150.0,visa,226.0,debit,315.0,87.0,10.0,gmail.com,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,4.0,1.0,379.0,62.0,8.0,63.0,8.0,63.0,60.0,63.0,T,T,T,M0,F,T,F,F,T,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,323.0,0.0,0.0,323.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,323.0,0.0,0.0,323.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.532599493153256
3504664,0,13559425,92.0,W,11204,298.0,150.0,visa,226.0,debit,272.0,87.0,8.0,charter.net,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,495.0,495.0,495.0,495.0,495.0,495.0,0.0,495.0,T,T,T,M0,F,F,T,T,T,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.

id,target,ps_ind_01,ps_ind_02_cat,ps_ind_03,ps_ind_04_cat,ps_ind_05_cat,ps_ind_06_bin,ps_ind_07_bin,ps_ind_08_bin,ps_ind_09_bin,ps_ind_10_bin,ps_ind_11_bin,ps_ind_12_bin,ps_ind_13_bin,ps_ind_14,ps_ind_15,ps_ind_16_bin,ps_ind_17_bin,ps_ind_18_bin,ps_reg_01,ps_reg_02,ps_reg_03,ps_car_01_cat,ps_car_02_cat,ps_car_03_cat,ps_car_04_cat,ps_car_05_cat,ps_car_06_cat,ps_car_07_cat,ps_car_08_cat,ps_car_09_cat,ps_car_10_cat,ps_car_11_cat,ps_car_11,ps_car_12,ps_car_13,ps_car_14,ps_car_15,ps_calc_01,ps_calc_02,ps_calc_03,ps_calc_04,ps_calc_05,ps_calc_06,ps_calc_07,ps_calc_08,ps_calc_09,ps_calc_10,ps_calc_11,ps_calc_12,ps_calc_13,ps_calc_14,ps_calc_15_bin,ps_calc_16_bin,ps_calc_17_bin,ps_calc_18_bin,ps_calc_19_bin,ps_calc_20_bin
7,0,2,2.0,5,1.0,0.0,0,1,0,0,0,0,0,0,0,11,0,1,0,0.7,0.2,0.7180703307999999,10.0,1,1.0,0,1.0,4,1.0,0,0.0,1,12,2,0.4,0.8836789178,0.3708099244,3.6055512755000003,0.6,0.5,0.2,3,1,10,1,10,1,5,9,1,5,8,0,1,1,0,0,1
13,0,5,4.0,9,1.0,0.0,0,0,1,0,0,0,0,0,0,12,1,0,0,0.0,0.0,0.8015609771,7.0,1,1.0,0,1.0,14,1.0,1,2.0,1,60,1,0.316227766,0.6415857163,0.34727510710000004,3.3166247904,0.5,0.7,0.1,2,2,9,1,8,2,7,4,2,7,7,0,1,1,0,1,0
17,0,0,2.0,0,1.0,0.0,1,0,0,0,0,0,0,0,0,9,1,0,0,0.7,0.6,0.840758586,11.0,1,1.0,0,1.0,14,1.0,1,2.0,1,82,3,0.3160696126,0.5658315025,0.3651027253,2.0,0.4,0.6,0.0,2,2,6,3,10,2,12,3,1,1,3,0,0,0,1,1,0
20,0,2,1.0,3,1.0,0.0,0,1,0,0,0,0,0,0,0,8,1,0,0,0.6,0.1,0.6174544518,6.0,1,1.0,0,1.0,11,1.0,1,0.0,1,99,2,0.316227766,0.6396829018,0.3687817783,3.1622776602,0.2,0.6,0.5,2,2,8,1,8,3,10,3,0,0,10,0,1,0,0,1,0
50,0,1,2.0,1,0.0,0.0,0,0,1,0,0,0,0,0,0,9,1,0,0,0.6,0.3,0.6995534290000001,11.0,1,1.0,1,1.0,3,1.0,1,0.0,1,59,3,0.316227766,0.7637040012999999,0.3498571137,3.6055512755000003,0.3,0.0,0.3,2,1,10,6,7,3,5,3,3,1,8,0,0,1,0,0,0
